# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a CroissantMetadata object

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below we list the available record sets, their names, and corresponding `@id`s, as well as their fields (`@id`s and names).

In [ ]:
# List all record sets and their fields using @id referencing as required

record_sets = list(metadata.recordSets)
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record set: {rs.name}")
    print(f"  @id: {rs.id}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field in rs.fields:
            print(f"   - {field.name} (@id: {field.id}, type: {field.dataType})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Choose the primary record set by @id (from the previous cell output)
# For the FAIR^2 dataset, the main table is typically 'recordSet:main' or similar.
main_record_set_id = None

# Let's programmatically find the largest table as main
max_field_count = -1
for rs in record_sets:
    if hasattr(rs, 'fields') and rs.fields and len(rs.fields) > max_field_count:
        main_record_set_id = rs.id
        max_field_count = len(rs.fields)

print(f"Using record set @id for main table: {main_record_set_id}\n")

# Extract all record set IDs
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    # Clean up DataFrame column names (they are field @id's by default)
    dataframes[record_set_id] = df

print(f"Columns for main record set ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, let's select a numeric field and a grouping field using @id.

# Find numeric fields (Float/Integer/Number)
main_record_set = None
for rs in record_sets:
    if rs.id == main_record_set_id:
        main_record_set = rs
        break

numeric_field_id = None
group_field_id = None
for field in main_record_set.fields:
    if str(field.dataType).lower() in ('float', 'number', 'integer') and numeric_field_id is None:
        numeric_field_id = field.id
    if ('group' in field.name.lower() or 'sample' in field.name.lower()) and not group_field_id:
        group_field_id = field.id

# If not found, use the first available
if not numeric_field_id:
    for field in main_record_set.fields:
        if str(field.dataType).lower() in ('float', 'number', 'integer'):
            numeric_field_id = field.id
            break
if not group_field_id and len(main_record_set.fields) >= 2:
    group_field_id = main_record_set.fields[1].id

print(f"Numeric field selected: {numeric_field_id}")
print(f"Grouping field selected: {group_field_id}\n")

df = dataframes[main_record_set_id].copy()

# Some fields may contain string representations, try to coerce
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].quantile(0.75)  # filter top 25% as an example
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

if group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id, dropna=False)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the selected numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# Boxplot grouped by group_field (if present)
if group_field_id in df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"Boxplot of {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we loaded the FAIR^2 dataset ("Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells") using `mlcroissant`.
* Explored available record sets, their IDs, and fields, referencing all entities via their `@id`.
* Extracted the main data table and performed example EDA steps such as filtering high-abundance proteins, normalizing values, and grouping by sample or experimental group.
* Visualized numeric protein characteristics to reveal data distributions and subgroup differences.

This workflow can be further extended for advanced statistical analysis, visualization, or FAIR evaluation based on the rich metadata provided with Croissant datasets.